In [1]:
import torch
import time
import pandas as pd
import os

from torchvision.models import mobilenet_v2

In [2]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu126
CUDA available: True


In [3]:
model = mobilenet_v2(weights="DEFAULT")
model.eval()

print("Model loaded.")

Model loaded.


In [4]:
import torch

quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("Quantization applied.")

Quantization applied.


C:\Users\user\AppData\Local\Temp\ipykernel_20336\1745826588.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [5]:
def benchmark(model, device, input_tensor, runs=100):

    model = model.to(device)
    input_tensor = input_tensor.to(device)

    # Warm-up runs
    for _ in range(10):
        with torch.no_grad():
            _ = model(input_tensor)

    # GPU synchronization
    if device == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    for _ in range(runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.time()

    avg_latency = (end - start) / runs

    return avg_latency

In [6]:
input_tensor = torch.randn(1, 3, 224, 224)

In [7]:
cpu_latency = benchmark(
    quantized_model,
    "cpu",
    input_tensor
)

print(f"CPU Average Latency: {cpu_latency:.6f} seconds")

CPU Average Latency: 0.066799 seconds


In [8]:
import os
import torch

torch.save(
    quantized_model.state_dict(),
    "quantized_model.pth"
)

size_mb = os.path.getsize(
    "quantized_model.pth"
) / (1024 * 1024)

print(f"Quantized model size: {size_mb:.2f} MB")

Quantized model size: 9.94 MB
